# Stream Basics


Stream Basics: stream() vs invoke() [Step 05 - Streaming Fundamentals]

> **MLCourse - Agentic AI - LangGraph**

LangGraph offers two ways to run a compiled graph: `invoke()` returns the
final state after the entire graph finishes, while `stream()` yields updates
as each node completes. This notebook builds a simple two-node graph and
demonstrates both approaches side-by-side so you can see exactly what
streaming reveals that invoke() hides.

# What you will learn

1. How to build a minimal LangGraph StateGraph with typed state.
2. The difference between `invoke()` (all-or-nothing) and `stream()` (incremental).
3. How `stream()` yields node-level updates in execution order.
4. Visualizing the compiled graph with Mermaid diagrams.

### Key takeaways

- `invoke()` blocks until the graph finishes and returns only the final state.
- `stream()` yields a dict for each step, giving you progress visibility.
- Both return the same final result; streaming is about *when* you see it.

### Setup: imports, LLM, environment


In [ ]:
import os                           # env access for API keys
from dotenv import load_dotenv      # load .env file if present

load_dotenv(override=False)         # load without overriding existing env vars

from typing import Annotated, TypedDict  # typed graph state

from langgraph.graph import StateGraph, END  # core graph primitives

from langchain_ollama import ChatOllama  # local LLM, no API key needed


### Model guard: ensure ChatOllama is reachable


In [ ]:
try:                                        # attempt a quick ping to ollama
    _test = ChatOllama(model="llama3.1:8b", temperature=0)  # lightweight model
    _test.invoke("ping")                    # will raise if ollama is down
    LLM_AVAILABLE = True                    # flag for downstream cells
    print("Ollama is reachable -- full notebook will run")
except Exception as exc:                    # connection refused or timeout
    LLM_AVAILABLE = False                   # disable LLM-dependent cells
    print("Ollama not reachable:", exc)
    print("LLM cells will be skipped; graph structure cells still run")


### Define graph state: a single messages list


In [ ]:
class GraphState(TypedDict):
    """Minimal state: just a list of messages flowing through the graph."""
    messages: Annotated[list, "conversation messages"]


### Define two simple nodes


In [ ]:
def node_a(state: GraphState) -> dict:
    """Node A: prepends a system-like message to the state."""
    print("[node_a] running -- adding greeting")   # progress indicator
    return {"messages": ["Hello from Node A"]}     # partial state update

def node_b(state: GraphState) -> dict:
    """Node B: appends a follow-up message after Node A."""
    print("[node_b] running -- adding follow-up")  # progress indicator
    return {"messages": ["Hello from Node B"]}     # partial state update


### Build the graph: A -> B -> END


In [ ]:
graph_builder = StateGraph(GraphState)     # instantiate with our state schema
graph_builder.add_node("a", node_a)        # register node_a as "a"
graph_builder.add_node("b", node_b)        # register node_b as "b"
graph_builder.set_entry_point("a")         # execution starts at node_a
graph_builder.add_edge("a", "b")           # after a finishes, run b
graph_builder.add_edge("b", END)           # after b finishes, stop

graph = graph_builder.compile()            # finalize the graph for execution

print("Graph compiled successfully")


### Visualize the graph


In [ ]:
from IPython.display import Image, display  # for Mermaid rendering

try:                                        # wrap in try/except for offline
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:                    # no internet or mermaid unavailable
    print("Graph visualization unavailable:", exc)
    print("Graph nodes:", list(graph_builder.nodes.keys()))


### invoke(): runs the whole graph, returns only final state


In [ ]:
initial = {"messages": ["start"]}          # seed state with one message

final_state = graph.invoke(initial)        # blocks until graph completes

print("--- invoke() result ---")            # section header
for msg in final_state["messages"]:        # iterate all accumulated messages
    print(" ", msg)                        # print each message


### stream(): yields updates as each node completes


In [ ]:
print("--- stream() output ---")            # section header

for step in graph.stream(initial):         # generator: yields per-node dicts
    for node_name, node_output in step.items():  # each step is {node: output}
        print("[step] node=%s" % node_name)       # which node just finished
        if node_output and "messages" in node_output:  # did it return messages?
            for m in node_output["messages"]:    # print each new message
                print("   ", m)


### Compare side-by-side: invoke vs stream


In [ ]:
print("=== Side-by-side comparison ===")
print()

# invoke: one shot, final state only
result_invoke = graph.invoke({"messages": ["compare"]})
print("[invoke] got %d messages total" % len(result_invoke["messages"]))

# stream: incremental visibility
step_count = 0                             # count how many steps we see
for step in graph.stream({"messages": ["compare"]}):
    step_count += 1                        # increment for each yielded step
    for node_name in step:                 # just count nodes
        print("[stream] step %d: node '%s' completed" % (step_count, node_name))

print("[stream] saw %d steps total" % step_count)
print()
print("Both approaches produce the same final state.")
print("Streaming gives you real-time progress; invoke gives you only the end.")


### Bonus: stream with output_keys to filter what you see


In [ ]:
print("=== Streaming with output_keys ===")

for step in graph.stream(initial, output_keys=["messages"]):
    # output_keys filters the yielded dict to only include specified keys
    for node_name, node_output in step.items():
        msgs = node_output.get("messages", [])  # safely get messages list
        print("[filtered] node=%s, new_msgs=%d" % (node_name, len(msgs)))

print()
print("NOTEBOOK COMPLETE: stream() vs invoke() demonstrated successfully")
